# <font color='003366'> **Entrega 2 - Proyecto Kaggle**

**Estudiante:** Juan Felipe Gutierrez Sanchez

**Cédula:**1000182850

## <font color='#1E90FF'> **Descripción**

Las Pruebas Saber Pro son evaluaciones estandarizadas aplicadas en Colombia con el propósito de medir la calidad, los conocimientos y las competencias de los estudiantes que cursan programas de educación superior, como universidades e instituciones tecnológicas.
Forman parte de las estrategias del Gobierno colombiano para evaluar y fortalecer la calidad de la educación en este nivel.

Estas pruebas incluyen cinco componentes genéricos: Inglés, Lectura Crítica, Competencias Ciudadanas, Razonamiento Cuantitativo y Comunicación Escrita.

El objetivo del trabajo consiste en desarrollar un modelo de clasificación capaz de predecir el nivel de desempeño de cada estudiante en estas pruebas, clasificándolo en una de las siguientes categorías: bajo, medio-bajo, medio-alto o alto.

## <font color='#1E90FF'> **1. Inicialización**

### <font color='46B8A9'> **1.1. Librerias**

In [10]:
import warnings
warnings.filterwarnings("ignore")

# Datos
import os
from google.colab import files
import pandas as pd
import numpy as np
from itertools import product

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

# Modelado
from sklearn.preprocessing import LabelEncoder, MinMaxScaler,StandardScaler

### <font color='46B8A9'> **1.2. Descarga y carga de datos desde Kaggle**

In [1]:
from google.colab import files
files.upload()

Saving kaggle (2).json to kaggle (2).json


{'kaggle (2).json': b'{"username":"jfelipegutierrez1","key":"242c6e3c228e9d9d5bc6321345ac3177"}'}

In [2]:
import os

# Crear la carpeta de configuración de Kaggle
!mkdir -p ~/.kaggle

# Mover y renombrar el archivo correctamente
!mv "kaggle (2).json" ~/.kaggle/kaggle.json

# Ajustar permisos
!chmod 600 ~/.kaggle/kaggle.json

# Verificar que el archivo quedó en el lugar correcto
!ls -la ~/.kaggle

total 16
drwxr-xr-x 2 root root 4096 Nov  3 02:31 .
drwx------ 1 root root 4096 Nov  3 02:31 ..
-rw------- 1 root root   73 Nov  3 02:31 kaggle.json


In [3]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()
print("✅ Kaggle autenticado correctamente")

✅ Kaggle autenticado correctamente


In [4]:
!kaggle competitions download -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia -p /content

  0% 0.00/29.9M [00:00<?, ?B/s]
100% 29.9M/29.9M [00:00<00:00, 791MB/s]


In [5]:
import zipfile

zip_path = '/content/udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/data')

print("✅ Archivos extraídos correctamente en '/content/data'")

✅ Archivos extraídos correctamente en '/content/data'


In [6]:
import os
print(os.listdir('/content/data'))

['train.csv', 'submission_example.csv', 'test.csv']


In [7]:

# Librerías para procesamiento
import pandas as pd
import numpy as np

df = pd.read_csv("/content/data/train.csv")


### <font color='46B8A9'> **1.3. Descripción general de los datos**

In [8]:
rows, cols = df.shape
print(f'Hay {rows} filas y {cols} columnas en el dataset')

Hay 692500 filas y 21 columnas en el dataset


In [11]:
# Mostrar información general del DataFrame
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 692500 entries, 0 to 692499
Data columns (total 21 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   ID                           692500 non-null  int64  
 1   PERIODO_ACADEMICO            692500 non-null  int64  
 2   E_PRGM_ACADEMICO             692500 non-null  object 
 3   E_PRGM_DEPARTAMENTO          692500 non-null  object 
 4   E_VALORMATRICULAUNIVERSIDAD  686213 non-null  object 
 5   E_HORASSEMANATRABAJA         661643 non-null  object 
 6   F_ESTRATOVIVIENDA            660363 non-null  object 
 7   F_TIENEINTERNET              665871 non-null  object 
 8   F_EDUCACIONPADRE             669322 non-null  object 
 9   F_TIENELAVADORA              652727 non-null  object 
 10  F_TIENEAUTOMOVIL             648877 non-null  object 
 11  E_PRIVADO_LIBERTAD           692500 non-null  object 
 12  E_PAGOMATRICULAPROPIO        686002 non-null  object 
 13 

## <font color='#1E90FF'> **2. Limpieza y preprocesado de datos**

### <font color='46B8A9'> **2.1. Identificación y eliminación de variables (columnas) iguales**

Se observa que en el conjunto de datos existen dos columnas con nombres muy similares: 'F_TIENEINTERNET' y 'F_TIENEINTERNET.1', lo cual sugiere que podrían contener la misma información.
Para confirmar esta hipótesis, se compararon ambas columnas utilizando el método .equals() de pandas, que permite verificar si todos los elementos de ambas columnas son idénticos.
Si el resultado obtenido es True, se concluye que las dos columnas son duplicadas y, por tanto, es recomendable eliminar una de ellas para evitar redundancia y reducir la dimensionalidad del conjunto de datos.

In [12]:
df1 = df.copy()  # Crea una copia del dataframe original

In [14]:
# Verificar si las columnas 'F_TIENEINTERNET' y 'F_TIENEINTERNET.1' contienen exactamente la misma información
df1['F_TIENEINTERNET'].equals(df1['F_TIENEINTERNET.1'])

True

In [15]:
# Eliminar columna 'F_TIENEINTERNET.1'
df1.drop('F_TIENEINTERNET.1', axis=1, inplace=True)

### <font color='46B8A9'> **2.2. Tratamiento de datos nulos**

En la entrega 1 se identificó que varias variables categóricas del conjunto de datos presentaban una pequeña proporción de valores nulos, en la mayoría de los casos inferior al 5% del total de registros. Dado que este porcentaje es bajo, no supone un impacto significativo en el análisis general.

Por esta razón, en lugar de eliminar registros o aplicar técnicas avanzadas de imputación, se decidió reemplazar los valores faltantes con la moda, es decir, con la categoría más frecuente dentro de cada variable. Esta decisión permite preservar la totalidad de las observaciones y mantener la coherencia con la distribución original de los datos.

Además, al tratarse de variables de tipo categórico, este método resulta adecuado y sencillo de implementar, ya que conserva la interpretabilidad del conjunto de datos y facilita las etapas posteriores del preprocesamiento y modelado.

In [16]:
# Calcular la cantidad de valores nulos por columna en el DataFrame
null_counts = df1.isnull().sum()

# Filtrar solo las columnas que tienen al menos un valor nulo
null_counts = null_counts[null_counts > 0]

# Mostrar el número de valores nulos por columna (solo las que tienen nulos)
null_counts

,0
E_VALORMATRICULAUNIVERSIDAD,6287
E_HORASSEMANATRABAJA,30857
F_ESTRATOVIVIENDA,32137
F_TIENEINTERNET,26629
F_EDUCACIONPADRE,23178
F_TIENELAVADORA,39773
F_TIENEAUTOMOVIL,43623
E_PAGOMATRICULAPROPIO,6498
F_TIENECOMPUTADOR,38103
F_EDUCACIONMADRE,23664


In [17]:

# Iterar sobre todas las columnas categóricas que tienen valores nulos
for column in null_counts.index:
    # Obtener la moda (valor más frecuente) de la columna
    mode_value = df1[column].mode()[0]
    # Reemplazar los valores nulos en la columna con la moda
    df1[column].fillna(mode_value, inplace=True)

# Calcular el total de valores nulos restantes en todo el DataFrame (debería dar 0 si se imputaron todos)
remaining_nulls = df1.isnull().sum().sum()
remaining_nulls

np.int64(0)

In [18]:

# Verificar presencia de nulos
df1.isnull().sum()

,0
ID,0
PERIODO_ACADEMICO,0
E_PRGM_ACADEMICO,0
E_PRGM_DEPARTAMENTO,0
E_VALORMATRICULAUNIVERSIDAD,0
E_HORASSEMANATRABAJA,0
F_ESTRATOVIVIENDA,0
F_TIENEINTERNET,0
F_EDUCACIONPADRE,0
F_TIENELAVADORA,0


### <font color='46B8A9'> **2.3. Variable ID**

In [19]:
# La siguiente línea modifica directamente el DataFrame df1, estableciendo la columna 'ID' como índice
df1.set_index('ID', inplace=True)

In [20]:
df1

,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,F_TIENEAUTOMOVIL,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_EDUCACIONMADRE,RENDIMIENTO_GLOBAL,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4
ID,,,,,,,,,,,,,,,,,,,
904256,20212,ENFERMERIA,BOGOTÁ,Entre 5.5 millones y menos de 7 millones,Menos de 10 horas,Estrato 3,Si,Técnica o tecnológica incompleta,Si,Si,N,No,Si,Postgrado,medio-alto,0.322,0.208,0.310,0.267
645256,20212,DERECHO,ATLANTICO,Entre 2.5 millones y menos de 4 millones,0,Estrato 3,No,Técnica o tecnológica completa,Si,No,N,No,Si,Técnica o tecnológica incompleta,bajo,0.311,0.215,0.292,0.264
308367,20203,MERCADEO Y PUBLICIDAD,BOGOTÁ,Entre 2.5 millones y menos de 4 millones,Más de 30 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,No,N,No,No,Secundaria (Bachillerato) completa,bajo,0.297,0.214,0.305,0.264
470353,20195,ADMINISTRACION DE EMPRESAS,SANTANDER,Entre 4 millones y menos de 5.5 millones,0,Estrato 4,Si,No sabe,Si,No,N,No,Si,Secundaria (Bachillerato) completa,alto,0.485,0.172,0.252,0.190
989032,20212,PSICOLOGIA,ANTIOQUIA,Entre 2.5 millones y menos de 4 millones,Entre 21 y 30 horas,Estrato 3,Si,Primaria completa,Si,Si,N,No,Si,Primaria completa,medio-bajo,0.316,0.232,0.285,0.294
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25096,20195,BIOLOGIA,LA GUAJIRA,Entre 500 mil y menos de 1 millón,Entre 11 y 20 horas,Estrato 2,Si,Secundaria (Bachillerato) completa,Si,No,N,Si,Si,Secundaria (Bachillerato) incompleta,medio-alto,0.237,0.271,0.271,0.311
754213,20212,PSICOLOGIA,NORTE SANTANDER,Entre 2.5 millones y menos de 4 millones,Más de 30 horas,Estrato 3,Si,Primaria incompleta,Si,No,N,No,Si,Secundaria (Bachillerato) incompleta,bajo,0.314,0.240,0.278,0.260
504185,20183,ADMINISTRACIÓN EN SALUD OCUPACIONAL,BOGOTÁ,Entre 1 millón y menos de 2.5 millones,Menos de 10 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,No,N,Si,Si,Secundaria (Bachillerato) incompleta,medio-bajo,0.286,0.240,0.314,0.287


### <font color='46B8A9'> **2.4. Variable PERIODO_ACADEMICO**

Se genera una nueva columna denominada ‘AÑO’ a partir de la variable ‘PERIODO_ACADEMICO’, ya que esta última contiene diversas categorías que representan periodos académicos específicos (por ejemplo, 20195 o 20212). Algunas de estas categorías concentran una cantidad de observaciones significativamente mayor que otras, lo que puede provocar desequilibrios en el análisis y en la construcción de los modelos.

Al extraer únicamente el año de cada periodo, se logra reducir la complejidad categórica de la variable y equilibrar la distribución de los datos, facilitando así un procesamiento más homogéneo y estable para las etapas posteriores del modelado.

In [21]:
df1.PERIODO_ACADEMICO.value_counts()

,count
PERIODO_ACADEMICO,
20195,180873
20203,171838
20212,171412
20183,164818
20194,1472
20213,1178
20202,490
20184,254
20196,165


In [22]:
'''
Se crea una nueva columna llamada 'AÑO' a partir de la variable 'PERIODO_ACADEMICO',
ya que algunas de sus categorías son muy grandes mientras que otras son demasiado pequeñas.
Agrupar los periodos por año permite simplificar la variable, facilitar la interpretación
de los resultados y captar tendencias generales a lo largo del tiempo sin perder información relevante.
'''

# Definir un diccionario para mapear los valores de 'PERIODO_ACADEMICO' a años
agrupacionPERIODO = {
    20195: 2019,
    20203: 2020,
    20212: 2021,
    20183: 2018,
    20194: 2019,
    20213: 2021,
    20202: 2020,
    20184: 2018,
    20196: 2019
}

# Aplicar el mapeo de 'PERIODO_ACADEMICO' a 'AÑO' usando el diccionario
df1['AÑO'] = df1['PERIODO_ACADEMICO'].map(agrupacionPERIODO)

In [23]:
df1.AÑO.value_counts()

,count
AÑO,
2019,182510
2021,172590
2020,172328
2018,165072


### <font color='46B8A9'> **2.5. Variable E_PRGM_ACADEMICO**

Con el fin de simplificar el análisis y mejorar la interpretabilidad de la información, las numerosas categorías originales de programas académicos se agrupan en categorías más amplias y representativas.

Esta agrupación se realiza mediante una función personalizada que clasifica cada programa según palabras clave presentes en su nombre, garantizando que programas similares queden integrados dentro de un mismo grupo general.

Esta estrategia contribuye a reducir la alta dimensionalidad derivada de manejar múltiples categorías individuales, al tiempo que mantiene la coherencia semántica entre los distintos programas académicos.

In [24]:
def clasificar_carrera(carrera):
    carrera = carrera.upper()

    # Ingenierías
    if 'INGENIER' in carrera or 'INGENIER¿' in carrera or 'INGENIERÌ' in carrera:
        return 'Ingeniería'

    # Ciencias de la Salud
    salud = ['MEDICINA', 'ENFERMER', 'ODONTOLOG', 'FISIOTERAP', 'FARMACIA',
             'NUTRICI', 'TERAPIA', 'OPTOMETR', 'BACTERIOLOG', 'BIOANALISIS',
             'INSTRUMENTACION QUIRURGICA', 'FONOAUDIOLOG', 'GERONTOLOG']
    if any(palabra in carrera for palabra in salud):
        return 'Ciencias de la Salud'

    # Ciencias Sociales y Humanidades
    sociales = ['DERECHO', 'PSICOLOG', 'SOCIOLOG', 'TRABAJO SOCIAL', 'CIENCIA POLITICA',
                'ANTROPOLOG', 'HISTORIA', 'FILOSOF', 'COMUNICACION', 'PERIODISMO']
    if any(palabra in carrera for palabra in sociales):
        return 'Ciencias Sociales y Humanidades'

    # Educación
    if 'EDUCACION' in carrera or 'EDUCACI¿N' in carrera or 'PEDAGOG' in carrera or 'LICENCIATURA' in carrera:
        return 'Educación'

    # Administración y Negocios
    admin = ['ADMINISTRACION', 'ADMINISTRACI¿N', 'ADMINISTRACIÒN', 'NEGOCIOS',
             'FINANZAS', 'CONTADUR', 'ECONOM', 'EMPRESA', 'MERCADEO', 'MARKETING',
             'COMERCIO', 'RELACIONES INTERNACIONALES', 'PUBLICIDAD']
    if any(palabra in carrera for palabra in admin):
        return 'Administración y Negocios'

    # Ciencias Básicas
    basicas = ['BIOLOG', 'QUIMICA', 'FISICA', 'MATEMATIC', 'ESTADISTIC', 'GEOLOG',
               'MICROBIOLOG', 'ECOLOG', 'CIENCIA', 'BIOQUIM']
    if any(palabra in carrera for palabra in basicas):
        return 'Ciencias Básicas'

    # Arte y Diseño
    arte = ['ARTE', 'DISEÑO', 'DISE¿O', 'MUSICA', 'TEATRO', 'DANZA', 'CINE',
            'BELLAS ARTES', 'GASTRONOM', 'CULINARIA']
    if any(palabra in carrera for palabra in arte):
        return 'Arte y Diseño'

    # Arquitectura y Urbanismo
    if 'ARQUITECTURA' in carrera or 'URBANISMO' in carrera:
        return 'Arquitectura y Urbanismo'

    # Agronomía y afines
    if 'AGRONOM' in carrera or 'ZOOTECN' in carrera or 'AGROPECUAR' in carrera or 'AGROINDUSTRIAL' in carrera:
        return 'Agronomía y Ciencias Agropecuarias'

    # Tecnología e Informática
    if 'TECNOLOG' in carrera or 'INFORMATICA' in carrera or 'SISTEMAS' in carrera:
        return 'Tecnología e Informática'

    # Si no coincide con ninguna categoría anterior
    return 'Otras'


# Aplicar la función a la columna actualizada
df1['PRGM_ACADEMICO'] = df1['E_PRGM_ACADEMICO'].apply(clasificar_carrera)

# Verificar las categorías creadas
df1['PRGM_ACADEMICO'].value_counts()

,count
PRGM_ACADEMICO,
Administración y Negocios,198410
Ingeniería,148431
Ciencias Sociales y Humanidades,141503
Educación,64442
Ciencias de la Salud,57211
Otras,37031
Arte y Diseño,16030
Ciencias Básicas,14856
Arquitectura y Urbanismo,11746


### <font color='46B8A9'> **2.6. Variable E_PRGM_DEPARTAMENTO**

Con el propósito de facilitar el análisis y reducir la alta cardinalidad de la variable E_PRGM_DEPARTAMENTO, los distintos departamentos del país se agrupan en seis grandes regiones geográficas de Colombia: Andina, Caribe, Pacífica, Orinoquía, Amazónica e Insular.

Esta regionalización permite simplificar la representación territorial de los datos, mantener una coherencia geográfica y mejorar la interpretabilidad de los resultados en los procesos de análisis y modelado.

In [25]:
df1['E_PRGM_DEPARTAMENTO'].unique()  # Muestra los valores únicos presentes en la columna

array(['BOGOTÁ', 'ATLANTICO', 'SANTANDER', 'ANTIOQUIA', 'HUILA', 'SUCRE',
       'CAQUETA', 'CUNDINAMARCA', 'BOLIVAR', 'TOLIMA', 'VALLE', 'QUINDIO',
       'RISARALDA', 'CORDOBA', 'META', 'LA GUAJIRA', 'BOYACA', 'NARIÑO',
       'CAUCA', 'NORTE SANTANDER', 'CESAR', 'PUTUMAYO', 'CALDAS',
       'MAGDALENA', 'CHOCO', 'CASANARE', 'ARAUCA', 'GUAVIARE', 'AMAZONAS',
       'VAUPES', 'SAN ANDRES'], dtype=object)

In [27]:
'''
Se crea un diccionario de regiones porque agrupar los departamentos en categorías geográficas más amplias
simplifica el análisis, reduce la cardinalidad de la variable y permite identificar patrones regionales.
A partir de este diccionario, se genera una nueva columna llamada 'REGION' en el DataFrame para incorporar esta agrupación.
'''

# Crear un diccionario de regiones
departamentos_regiones = {
    # Región Andina
    'BOGOTÁ': 'Andina',
    'SANTANDER': 'Andina',
    'ANTIOQUIA': 'Andina',
    'HUILA': 'Andina',
    'CUNDINAMARCA': 'Andina',
    'TOLIMA': 'Andina',
    'QUINDIO': 'Andina',
    'RISARALDA': 'Andina',
    'BOYACA': 'Andina',
    'NORTE SANTANDER': 'Andina',
    'CALDAS': 'Andina',

    # Región Amazónica
    'CAQUETA': 'Amazónica',
    'PUTUMAYO': 'Amazónica',
    'GUAVIARE': 'Amazónica',
    'AMAZONAS': 'Amazónica',
    'VAUPES': 'Amazónica',

    # Región Pacífica
    'VALLE': 'Pacífica',
    'NARIÑO': 'Pacífica',
    'CAUCA': 'Pacífica',
    'CHOCO': 'Pacífica',

    # Región Caribe
    'ATLANTICO': 'Caribe',
    'SUCRE': 'Caribe',
    'BOLIVAR': 'Caribe',
    'CORDOBA': 'Caribe',
    'LA GUAJIRA': 'Caribe',
    'CESAR': 'Caribe',
    'MAGDALENA': 'Caribe',

    # Región Orinoquía
    'META': 'Orinoquía',
    'CASANARE': 'Orinoquía',
    'ARAUCA': 'Orinoquía',

    # Región Insular
    'SAN ANDRES': 'Insular'
}

# Asignar cada departamento a su región correspondiente
df1['REGION'] = df1['E_PRGM_DEPARTAMENTO'].map(departamentos_regiones)


In [28]:
df1.REGION.value_counts()

,count
REGION,
Andina,499712
Caribe,105080
Pacífica,73802
Orinoquía,10351
Amazónica,3545
Insular,10


### <font color='46B8A9'> **2.7. Variable E_VALORMATRICULAUNIVERSIDAD**

La variable E_VALORMATRICULAUNIVERSIDAD contiene categorías expresadas como rangos de valores monetarios.
Para hacerla más adecuada para su uso en modelos de Machine Learning, se transforma en una variable numérica, asignando a cada rango un valor representativo basado en el promedio del intervalo correspondiente.

Esta conversión permite preservar el orden y la magnitud relativa de los valores, lo cual es especialmente útil para modelos que capturan relaciones ordinales o numéricas, como la regresión logística, los árboles de decisión o los modelos de ensamble.

Además, al convertir esta variable a formato numérico se evita la expansión en múltiples columnas que generaría una codificación one-hot, reduciendo así la dimensionalidad del conjunto de datos.
En conjunto, esta transformación optimiza el procesamiento y puede mejorar el desempeño predictivo si existe una relación significativa entre el valor de la matrícula y el rendimiento académico del estudiante.

In [29]:
df1['E_VALORMATRICULAUNIVERSIDAD'].value_counts()

,count
E_VALORMATRICULAUNIVERSIDAD,
Entre 1 millón y menos de 2.5 millones,210335
Entre 2.5 millones y menos de 4 millones,127430
Menos de 500 mil,80263
Entre 500 mil y menos de 1 millón,78704
Entre 4 millones y menos de 5.5 millones,69736
Más de 7 millones,68014
Entre 5.5 millones y menos de 7 millones,38490
No pagó matrícula,19528


In [31]:
# Se asigna el valor promedio de pago de matrícula a cada categoría dentro de la variable
valormat = {
    'Entre 1 millón y menos de 2.5 millones': 1.75,
    'Entre 2.5 millones y menos de 4 millones': 3.25,
    'Menos de 500 mil': 0.25,
    'Entre 500 mil y menos de 1 millón': 0.75,
    'Entre 4 millones y menos de 5.5 millones': 4.75,
    'Más de 7 millones': 7.75,
    'Entre 5.5 millones y menos de 7 millones': 6.25,
    'No pagó matrícula': 0
}

# Usar map para transformar los valores a su equivalente numérico
df1['E_VALORMATRICULAUNIVERSIDAD'] = df1['E_VALORMATRICULAUNIVERSIDAD'].map(valormat)

# Contar los valores únicos después de la transformación
df1['E_VALORMATRICULAUNIVERSIDAD'].value_counts()

,count
E_VALORMATRICULAUNIVERSIDAD,
1.75,210335
3.25,127430
0.25,80263
0.75,78704
4.75,69736
7.75,68014
6.25,38490
0.00,19528


### <font color='46B8A9'> **2.8 Variable E_HORASSEMANATRABAJA**

Con el objetivo de convertir la variable E_HORASSEMANATRABAJA a un formato numérico adecuado para el análisis estadístico y la construcción de modelos de aprendizaje supervisado, se asigna a cada categoría un valor promedio representativo de las horas trabajadas semanalmente.

Esta transformación permite un tratamiento cuantitativo más preciso de la información, mejora la interpretabilidad de los datos y facilita que los modelos identifiquen la posible relación entre la cantidad de horas de trabajo y el rendimiento global del estudiante.

Además, al expresar la variable en una escala numérica continua, se optimiza el desempeño de los algoritmos que dependen de la magnitud y orden de los valores, como los modelos lineales, los árboles de decisión y los métodos de ensamble.

In [32]:
df1['E_HORASSEMANATRABAJA'].value_counts()


,count
E_HORASSEMANATRABAJA,
Más de 30 horas,280209
0,116550
Entre 11 y 20 horas,115857
Entre 21 y 30 horas,92693
Menos de 10 horas,87191


In [33]:
# Se asigna el valor promedio de horas a cada categoría dentro de la variable
horasem = {
    '0': 0,
    'Menos de 10 horas': 5,
    'Entre 11 y 20 horas': 15.5,
    'Entre 21 y 30 horas': 25.5,
    'Más de 30 horas': 35.5
}

# Usar map para transformar los valores a formato numérico
df1['E_HORASSEMANATRABAJA'] = df1['E_HORASSEMANATRABAJA'].map(horasem)

# Contar los valores únicos después de la transformación
df1['E_HORASSEMANATRABAJA'].value_counts()

,count
E_HORASSEMANATRABAJA,
35.5,280209
0.0,116550
15.5,115857
25.5,92693
5.0,87191


### <font color='46B8A9'> **2.9. Variable FAMI_ESTRATOVIVIENDA**

Se transforma la variable 'FAMI_ESTRATOVIVIENDA' para mejorar su utilidad en los modelos de aprendizaje supervisado. Se reemplazan los nombres de los estratos por sus respectivos valores numéricos, preservando así su naturaleza ordinal (es decir, una jerarquía de niveles socioeconómicos). Esta conversión permite que los modelos interpreten correctamente la relación de orden entre los estratos y facilita el procesamiento matemático de la variable.

In [34]:
df1['F_ESTRATOVIVIENDA'].value_counts()

,count
F_ESTRATOVIVIENDA,
Estrato 2,264808
Estrato 3,210685
Estrato 1,111991
Estrato 4,65514
Estrato 5,23608
Estrato 6,12605
Sin Estrato,3289


In [35]:
# Se reemplazan los nombres de los estratos por sus valores numéricos, manteniendo su naturaleza ordinal.
df1['F_ESTRATOVIVIENDA'] = df1['F_ESTRATOVIVIENDA'].replace({
    'Estrato 6': 6,
    'Estrato 1': 1,
    'Estrato 2': 2,
    'Estrato 3': 3,
    'Estrato 4': 4,
    'Estrato 5': 5
})

# Verificar la distribución de los valores después de la transformación
df1['F_ESTRATOVIVIENDA'].value_counts()

,count
F_ESTRATOVIVIENDA,
2,264808
3,210685
1,111991
4,65514
5,23608
6,12605
Sin Estrato,3289


### <font color='46B8A9'> **2.10. Variables F_EDUCACIONPADRE y F_EDUCACIONMADRE**

Con el fin de homogeneizar la información y mejorar la calidad de los datos, se unifican los valores inciertos de las variables F_EDUCACIONPADRE y F_EDUCACIONMADRE, reemplazando las categorías “No sabe” y “No aplica” por una única categoría denominada “Indeterminado”.

Esta transformación permite simplificar el conjunto de datos, reducir la cardinalidad de las variables y evitar la dispersión de información faltante en etiquetas que representan esencialmente el mismo estado de desconocimiento.

Como resultado, se obtiene una representación más coherente y manejable de los niveles educativos de los padres, lo que facilita la interpretación de los modelos de aprendizaje supervisado y contribuye a una mejor consistencia en el análisis.

In [36]:
df1['F_EDUCACIONPADRE'].value_counts()

,count
F_EDUCACIONPADRE,
Secundaria (Bachillerato) completa,151467
Primaria incompleta,125675
Educación profesional completa,83117
Secundaria (Bachillerato) incompleta,71654
Técnica o tecnológica completa,62995
Primaria completa,55958
Postgrado,44169
Educación profesional incompleta,27084
Técnica o tecnológica incompleta,22552


In [37]:
df1['F_EDUCACIONMADRE'].value_counts()

,count
F_EDUCACIONMADRE,
Secundaria (Bachillerato) completa,165408
Primaria incompleta,99420
Técnica o tecnológica completa,89542
Educación profesional completa,85326
Secundaria (Bachillerato) incompleta,81012
Primaria completa,56125
Postgrado,46246
Técnica o tecnológica incompleta,27533
Educación profesional incompleta,22470


In [38]:
'''
Se reemplazan las categorías 'No sabe' y 'No Aplica' por 'Indeterminado' en las columnas de educación
de madre y padre, para unificar los valores inciertos.
'''

# Reemplazar valores inciertos en la educación de la madre
df1['F_EDUCACIONMADRE'] = [
    'Indeterminado' if i in ['No sabe', 'No Aplica'] else i
    for i in df1['F_EDUCACIONMADRE'].values
]

# Reemplazar valores inciertos en la educación del padre
df1['F_EDUCACIONPADRE'] = [
    'Indeterminado' if i in ['No sabe', 'No Aplica'] else i
    for i in df1['F_EDUCACIONPADRE'].values
]


In [39]:
df1['F_EDUCACIONPADRE'].value_counts()


,count
F_EDUCACIONPADRE,
Secundaria (Bachillerato) completa,151467
Primaria incompleta,125675
Educación profesional completa,83117
Secundaria (Bachillerato) incompleta,71654
Técnica o tecnológica completa,62995
Primaria completa,55958
Postgrado,44169
Educación profesional incompleta,27084
Indeterminado,25821


In [40]:
df1['F_EDUCACIONMADRE'].value_counts()

,count
F_EDUCACIONMADRE,
Secundaria (Bachillerato) completa,165408
Primaria incompleta,99420
Técnica o tecnológica completa,89542
Educación profesional completa,85326
Secundaria (Bachillerato) incompleta,81012
Primaria completa,56125
Postgrado,46246
Técnica o tecnológica incompleta,27533
Educación profesional incompleta,22470


## <font color='#1E90FF'> **3. Selección de variables y preparación de los datos**

### <font color='46B8A9'> **3.1. Selección de variables**

In [41]:
# Mostrar información general del DataFrame
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 692500 entries, 904256 to 933374
Data columns (total 22 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   PERIODO_ACADEMICO            692500 non-null  int64  
 1   E_PRGM_ACADEMICO             692500 non-null  object 
 2   E_PRGM_DEPARTAMENTO          692500 non-null  object 
 3   E_VALORMATRICULAUNIVERSIDAD  692500 non-null  float64
 4   E_HORASSEMANATRABAJA         692500 non-null  float64
 5   F_ESTRATOVIVIENDA            692500 non-null  object 
 6   F_TIENEINTERNET              692500 non-null  object 
 7   F_EDUCACIONPADRE             692500 non-null  object 
 8   F_TIENELAVADORA              692500 non-null  object 
 9   F_TIENEAUTOMOVIL             692500 non-null  object 
 10  E_PRIVADO_LIBERTAD           692500 non-null  object 
 11  E_PAGOMATRICULAPROPIO        692500 non-null  object 
 12  F_TIENECOMPUTADOR            692500 non-null  object 
 13 

Análisis de correlación entre variables numéricas y variable respuesta

Para analizar la relación entre las variables predictoras (E_VALORMATRICULAUNIVERSIDAD, E_HORASSEMANATRABAJA, INDICADOR_1, INDICADOR_2, INDICADOR_3, INDICADOR_4, AÑO) y la variable respuesta (RENDIMIENTO_GLOBAL), se empleó el coeficiente de correlación Eta (η).

Este coeficiente es especialmente adecuado cuando la variable dependiente es categórica y las variables independientes son numéricas, ya que permite cuantificar la fuerza de asociación entre ambas. Sus valores oscilan entre 0 (sin relación) y 1 (asociación perfecta).

El uso del coeficiente Eta facilita la identificación de las variables con mayor capacidad explicativa sobre el desempeño académico global, aportando información clave para la selección de variables relevantes en los modelos de aprendizaje supervisado.

In [42]:
# 1. Función para calcular el coeficiente de correlación tipo Eta
def correlation_ratio(categories, measurements):
    """
    categories: array-like categórico
    measurements: array-like numérico
    devuelve: eta, valor entre 0 y 1 que mide la asociación
    """
    # Convertir categorías a enteros 0, 1, 2, ...
    fcat, _ = pd.factorize(categories)
    cats = np.unique(fcat)
    mean_total = np.nanmean(measurements)

    # Suma de cuadrados entre categorías
    ss_between = sum(
        len(measurements[fcat == cat]) *
        (np.nanmean(measurements[fcat == cat]) - mean_total)**2
        for cat in cats
    )

    # Suma total de cuadrados
    ss_total = np.nansum((measurements - mean_total)**2)

    # Eta = sqrt(SS_between / SS_total)
    return np.sqrt(ss_between / ss_total) if ss_total > 0 else np.nan


# 2. Cálculo de η para cada variable numérica
eta_values = {}
for col in ['E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA',
            'INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4', 'AÑO']:
    eta = correlation_ratio(df1['RENDIMIENTO_GLOBAL'], df1[col].values)
    eta_values[col] = eta

# 3. Mostrar resultados ordenados
eta_df = (
    pd.DataFrame.from_dict(eta_values, orient='index', columns=['eta'])
    .sort_values(by='eta', ascending=False)
)
eta_df


,eta
E_VALORMATRICULAUNIVERSIDAD,0.269710
INDICADOR_1,0.252061
INDICADOR_2,0.195314
INDICADOR_4,0.135324
E_HORASSEMANATRABAJA,0.131308
INDICADOR_3,0.060940
AÑO,0.059160


En este caso, la variable E_VALORMATRICULAUNIVERSIDAD presentó la mayor correlación con la variable respuesta, seguida de INDICADOR_1, INDICADOR_2, INDICADOR_4 y E_HORASSEMANATRABAJA.

Este resultado sugiere que dichas variables podrían tener un peso predictivo significativo en la estimación del rendimiento global del estudiante, por lo que resultan especialmente relevantes para la etapa de selección de variables en los modelos de aprendizaje supervisado.

Análisis de correlación entre variables categóricas y la variable respuesta

Para determinar qué variables categóricas presentan una relación estadísticamente significativa con la variable de respuesta RENDIMIENTO_GLOBAL, se aplicó la prueba de Chi-cuadrado de independencia.

Esta prueba permite evaluar si existe una asociación entre dos variables categóricas, comparando la frecuencia observada con la esperada bajo el supuesto de independencia.
Al calcular el estadístico Chi² y su correspondiente p-valor para cada variable, es posible identificar cuáles características del conjunto de datos están significativamente asociadas con el desempeño académico de los estudiantes.

Este análisis resulta fundamental dentro del proceso de selección de variables, ya que permite reducir la dimensionalidad del modelo al eliminar variables no informativas y conservar aquellas que aportan valor predictivo relevante para la estimación del rendimiento global.

In [43]:
from scipy.stats import chi2_contingency

chi2_results = []

# 1. Seleccionar las columnas categóricas (excluyendo la variable objetivo)
categorical_columns = df1.columns[df1.dtypes == 'object'].tolist()
categorical_columns.remove('RENDIMIENTO_GLOBAL')

# 2. Aplicar la prueba Chi-cuadrado de independencia
for column in categorical_columns:
    # Crear tabla de contingencia entre la variable categórica y la variable respuesta
    contingency_table = pd.crosstab(df1[column], df1['RENDIMIENTO_GLOBAL'])

    # Calcular el estadístico Chi², el p-valor y los grados de libertad
    chi2, p_value, _, _ = chi2_contingency(contingency_table)

    # Guardar resultados
    chi2_results.append((column, chi2, p_value))

# 3. Convertir los resultados en un DataFrame y ordenar por significancia (p-valor)
chi2_results_df = pd.DataFrame(chi2_results, columns=['Variable', 'Chi2', 'p-value'])
chi2_results_df.sort_values(by='p-value', inplace=True)

# 4. Mostrar resultados
chi2_results_df


,Variable,Chi2,p-value
0,E_PRGM_ACADEMICO,143741.141869,0.000000
1,E_PRGM_DEPARTAMENTO,27731.258060,0.000000
2,F_ESTRATOVIVIENDA,54872.985913,0.000000
3,F_TIENEINTERNET,14122.196556,0.000000
4,F_EDUCACIONPADRE,57680.731917,0.000000
5,F_TIENELAVADORA,7823.648464,0.000000
6,F_TIENEAUTOMOVIL,19588.204011,0.000000
8,E_PAGOMATRICULAPROPIO,28125.037722,0.000000
12,REGION,8463.446239,0.000000
9,F_TIENECOMPUTADOR,12320.133185,0.000000


A partir de los resultados de la prueba de Chi-cuadrado, se puede interpretar la relación entre cada variable categórica y la variable de respuesta RENDIMIENTO_GLOBAL evaluando el p-valor asociado a cada una.

Interpretación de los resultados:

p-valor ≤ 0.05: La variable tiene una asociación estadísticamente significativa con RENDIMIENTO_GLOBAL. Es decir, resulta relevante para el análisis y podría contribuir al modelo predictivo.

p-valor > 0.05: No existe evidencia suficiente para afirmar una relación entre la variable y RENDIMIENTO_GLOBAL, por lo que puede considerarse irrelevante para efectos de predicción.

La variable E_PRIVADO_LIBERTAD presentó un p-valor = 0.269961, lo cual indica que no hay evidencia estadística suficiente para afirmar que esté asociada al rendimiento académico.
Por tanto, puede excluirse del análisis o de los modelos predictivos posteriores.

In [44]:

df1.columns

Index(['PERIODO_ACADEMICO', 'E_PRGM_ACADEMICO', 'E_PRGM_DEPARTAMENTO',
       'E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA',
       'F_ESTRATOVIVIENDA', 'F_TIENEINTERNET', 'F_EDUCACIONPADRE',
       'F_TIENELAVADORA', 'F_TIENEAUTOMOVIL', 'E_PRIVADO_LIBERTAD',
       'E_PAGOMATRICULAPROPIO', 'F_TIENECOMPUTADOR', 'F_EDUCACIONMADRE',
       'RENDIMIENTO_GLOBAL', 'INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3',
       'INDICADOR_4', 'AÑO', 'PRGM_ACADEMICO', 'REGION'],
      dtype='object')

In [45]:
# Se elimina la variable respuesta y seis variables que no se usarán en el modelo
X_features = df1.drop([
    'RENDIMIENTO_GLOBAL',
    'PERIODO_ACADEMICO',
    'E_PRGM_ACADEMICO',
    'E_PRGM_DEPARTAMENTO',
    'E_PRIVADO_LIBERTAD',
    'INDICADOR_3',
    'AÑO'
], axis=1, inplace=False)

In [46]:

X_features.dtypes

,0
E_VALORMATRICULAUNIVERSIDAD,float64
E_HORASSEMANATRABAJA,float64
F_ESTRATOVIVIENDA,object
F_TIENEINTERNET,object
F_EDUCACIONPADRE,object
F_TIENELAVADORA,object
F_TIENEAUTOMOVIL,object
E_PAGOMATRICULAPROPIO,object
F_TIENECOMPUTADOR,object
F_EDUCACIONMADRE,object


### <font color='46B8A9'> **3.2. Transformación de variables categóricas**

In [47]:
# Se identifican las columnas que son binarias (es decir, que tienen exactamente dos categorías distintas)
columnas_binarias = [col for col in X_features.columns if X_features[col].nunique() == 2]
columnas_binarias


['F_TIENEINTERNET',
 'F_TIENELAVADORA',
 'F_TIENEAUTOMOVIL',
 'E_PAGOMATRICULAPROPIO',
 'F_TIENECOMPUTADOR']

**Codificación One-Hot simplificada para variables categóricas binarias:**

In [48]:
# Aplicar codificación One-Hot simplificada (drop_first=True) a variables categóricas binarias
X_features = pd.get_dummies(
    X_features,
    columns=columnas_binarias,
    drop_first=True  # Evita la multicolinealidad generando solo una columna por variable (1 = 'Sí', 0 = 'No')
)

X_features.head()


,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_EDUCACIONPADRE,F_EDUCACIONMADRE,INDICADOR_1,INDICADOR_2,INDICADOR_4,PRGM_ACADEMICO,REGION,F_TIENEINTERNET_Si,F_TIENELAVADORA_Si,F_TIENEAUTOMOVIL_Si,E_PAGOMATRICULAPROPIO_Si,F_TIENECOMPUTADOR_Si
ID,,,,,,,,,,,,,,,
904256,6.25,5.0,3,Técnica o tecnológica incompleta,Postgrado,0.322,0.208,0.267,Ciencias de la Salud,Andina,True,True,True,False,True
645256,3.25,0.0,3,Técnica o tecnológica completa,Técnica o tecnológica incompleta,0.311,0.215,0.264,Ciencias Sociales y Humanidades,Caribe,False,True,False,False,True
308367,3.25,35.5,3,Secundaria (Bachillerato) completa,Secundaria (Bachillerato) completa,0.297,0.214,0.264,Administración y Negocios,Andina,True,True,False,False,False
470353,4.75,0.0,4,Indeterminado,Secundaria (Bachillerato) completa,0.485,0.172,0.190,Administración y Negocios,Andina,True,True,False,False,True
989032,3.25,25.5,3,Primaria completa,Primaria completa,0.316,0.232,0.294,Ciencias Sociales y Humanidades,Andina,True,True,True,False,True


**Codificación One-Hot Encoding para variables categóricas multiclase:**

In [49]:
# Aplicar One-Hot Encoding (para variables con 3 o más categorías)
# Lista de variables categóricas a transformar (nombres actualizados)
ohe_vars = ['F_ESTRATOVIVIENDA', 'F_EDUCACIONPADRE', 'F_EDUCACIONMADRE', 'PRGM_ACADEMICO', 'REGION']

# Generar las variables dummies para las columnas seleccionadas
X_features = pd.get_dummies(X_features, columns=ohe_vars)

# Visualizar las primeras filas del nuevo DataFrame
X_features.head()


,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,INDICADOR_1,INDICADOR_2,INDICADOR_4,F_TIENEINTERNET_Si,F_TIENELAVADORA_Si,F_TIENEAUTOMOVIL_Si,E_PAGOMATRICULAPROPIO_Si,F_TIENECOMPUTADOR_Si,...,PRGM_ACADEMICO_Educación,PRGM_ACADEMICO_Ingeniería,PRGM_ACADEMICO_Otras,PRGM_ACADEMICO_Tecnología e Informática,REGION_Amazónica,REGION_Andina,REGION_Caribe,REGION_Insular,REGION_Orinoquía,REGION_Pacífica
ID,,,,,,,,,,,,,,,,,,,,,
904256,6.25,5.0,0.322,0.208,0.267,True,True,True,False,True,...,False,False,False,False,False,True,False,False,False,False
645256,3.25,0.0,0.311,0.215,0.264,False,True,False,False,True,...,False,False,False,False,False,False,True,False,False,False
308367,3.25,35.5,0.297,0.214,0.264,True,True,False,False,False,...,False,False,False,False,False,True,False,False,False,False
470353,4.75,0.0,0.485,0.172,0.190,True,True,False,False,True,...,False,False,False,False,False,True,False,False,False,False
989032,3.25,25.5,0.316,0.232,0.294,True,True,True,False,True,...,False,False,False,False,False,True,False,False,False,False


Aunque varios algoritmos de machine learning pueden manejar valores booleanos directamente, convertirlos explícitamente a enteros (0 y 1) evita posibles ambigüedades, asegura la compatibilidad con todas las librerías de modelado y facilita tanto el análisis como la visualización de los datos.

Además, esta conversión contribuye a mantener la coherencia en los tipos de datos dentro del conjunto, especialmente al combinar variables booleanas con otras de naturaleza numérica.

In [50]:
# Convertir columnas booleanas a enteros explícitamente
for col in X_features.columns:
    if X_features[col].dtype == bool:
        X_features[col] = X_features[col].astype(int)

X_features.head()


,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,INDICADOR_1,INDICADOR_2,INDICADOR_4,F_TIENEINTERNET_Si,F_TIENELAVADORA_Si,F_TIENEAUTOMOVIL_Si,E_PAGOMATRICULAPROPIO_Si,F_TIENECOMPUTADOR_Si,...,PRGM_ACADEMICO_Educación,PRGM_ACADEMICO_Ingeniería,PRGM_ACADEMICO_Otras,PRGM_ACADEMICO_Tecnología e Informática,REGION_Amazónica,REGION_Andina,REGION_Caribe,REGION_Insular,REGION_Orinoquía,REGION_Pacífica
ID,,,,,,,,,,,,,,,,,,,,,
904256,6.25,5.0,0.322,0.208,0.267,1,1,1,0,1,...,0,0,0,0,0,1,0,0,0,0
645256,3.25,0.0,0.311,0.215,0.264,0,1,0,0,1,...,0,0,0,0,0,0,1,0,0,0
308367,3.25,35.5,0.297,0.214,0.264,1,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
470353,4.75,0.0,0.485,0.172,0.190,1,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0
989032,3.25,25.5,0.316,0.232,0.294,1,1,1,0,1,...,0,0,0,0,0,1,0,0,0,0


### <font color='46B8A9'> **3.3. Normalización de variables numéricas (Z-score)**

In [51]:
from sklearn.preprocessing import StandardScaler

# Almacenar variables numéricas (nombres actualizados)
numcol = ['E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA',
          'INDICADOR_1', 'INDICADOR_2', 'INDICADOR_4']

# Escalamiento con Z-Score
scaler = StandardScaler()
for col in numcol:
    X_features[[col]] = scaler.fit_transform(X_features[[col]])

X_features


,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,INDICADOR_1,INDICADOR_2,INDICADOR_4,F_TIENEINTERNET_Si,F_TIENELAVADORA_Si,F_TIENEAUTOMOVIL_Si,E_PAGOMATRICULAPROPIO_Si,F_TIENECOMPUTADOR_Si,...,PRGM_ACADEMICO_Educación,PRGM_ACADEMICO_Ingeniería,PRGM_ACADEMICO_Otras,PRGM_ACADEMICO_Tecnología e Informática,REGION_Amazónica,REGION_Andina,REGION_Caribe,REGION_Insular,REGION_Orinoquía,REGION_Pacífica
ID,,,,,,,,,,,,,,,,,,,,,
904256,1.488838,-1.133390,0.437002,-0.556223,0.060296,1,1,1,0,1,...,0,0,0,0,0,1,0,0,0,0
645256,0.182581,-1.487564,0.346934,-0.481341,0.016142,0,1,0,0,1,...,0,0,0,0,0,0,1,0,0,0
308367,0.182581,1.027071,0.232301,-0.492038,0.016142,1,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
470353,0.835710,-1.487564,1.771650,-0.941332,-1.072993,1,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0
989032,0.182581,0.318723,0.387874,-0.299484,0.457683,1,1,1,0,1,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25096,-0.905967,-0.389625,-0.258980,0.117717,0.707890,1,1,0,1,1,...,0,0,0,0,0,0,1,0,0,0
754213,0.182581,1.027071,0.371498,-0.213904,-0.042730,1,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0
504185,-0.470548,-1.133390,0.142233,-0.213904,0.354657,1,1,0,1,1,...,0,0,1,0,0,1,0,0,0,0


### <font color='46B8A9'> **3.4. Codificación y distribución de la variable objetivo**

Antes de implementar cualquier modelo predictivo, es fundamental preparar adecuadamente la variable objetivo RENDIMIENTO_GLOBAL.
Esta columna contiene etiquetas categóricas que representan distintos niveles de desempeño académico (por ejemplo: bajo, medio-bajo, medio-alto y alto).

Por esta razón, se realiza una conversión a valores numéricos discretos mediante un mapeo ordenado, que conserva la jerarquía natural entre las categorías.
Esta transformación permite que los modelos interpreten correctamente la estructura ordinal de la variable, facilitando tanto el entrenamiento del clasificador como la evaluación de su desempeño.

In [52]:
df1['RENDIMIENTO_GLOBAL'].value_counts()


,count
RENDIMIENTO_GLOBAL,
alto,175619
bajo,172987
medio-bajo,172275
medio-alto,171619


In [53]:
# Separar la variable respuesta (dependiente)
y_target = df1['RENDIMIENTO_GLOBAL']  # Variable a predecir

# Diccionario para transformar los valores categóricos en numéricos (manteniendo su orden jerárquico)
y_target_map = {
    'bajo': 0,
    'medio-bajo': 1,
    'medio-alto': 2,
    'alto': 3
}

# Aplicar la transformación directamente a la Serie y_target
y_target = y_target.map(y_target_map)

# Verificar la distribución de clases resultante
y_target.value_counts()


,count
RENDIMIENTO_GLOBAL,
3,175619
0,172987
1,172275
2,171619


In [54]:
# Gráfico variable respuesta a partir de y_target
counts = y_target.value_counts().reset_index()
counts.columns = ['RENDIMIENTO_GLOBAL', 'Count']  # Renombrar las columnas

# Calcular porcentaje y etiqueta
counts['Percentage'] = (counts['Count'] / counts['Count'].sum()) * 100
counts['Label'] = counts['Count'].astype(str) + ' (' + counts['Percentage'].round(1).astype(str) + '%)'

# Crear gráfico de barras interactivo
fig = px.bar(
    counts,
    x='RENDIMIENTO_GLOBAL',
    y='Count',
    color='RENDIMIENTO_GLOBAL',
    title='Distribución de RENDIMIENTO_GLOBAL',
    labels={'RENDIMIENTO_GLOBAL': 'Rendimiento Global', 'Count': 'Frecuencia'},
    text='Label',
    width=1000,
    height=600
)

fig.show()


In [55]:
counts

,RENDIMIENTO_GLOBAL,Count,Percentage,Label
0,3,175619,25.360144,175619 (25.4%)
1,0,172987,24.980072,172987 (25.0%)
2,1,172275,24.877256,172275 (24.9%)
3,2,171619,24.782527,171619 (24.8%)


In [56]:

y_target

,RENDIMIENTO_GLOBAL
ID,
904256,2
645256,0
308367,0
470353,3
989032,1
...,...
25096,2
754213,0
504185,1
